In [ ]:
# -----------------------------
# 0. Install Libraries
# -----------------------------
!pip install pandas gensim nltk scikit-learn gtts


In [ ]:
# -----------------------------
# 1. Import Libraries
# -----------------------------
import pandas as pd
import nltk
import numpy as np
import os

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from gtts import gTTS
from IPython.display import Audio, display

nltk.download("punkt")
nltk.download("punkt_tab")

# -----------------------------
# 2. Create AWS Question Answer Dataset
# -----------------------------
data = {
    "question": [
        "What is AWS EC2?",
        "What is AWS S3?",
        "What is AWS Lambda?",
        "What is IAM in AWS?",
        "What is Cloud Computing?",
        "What is Python used for?",
        "What is Machine Learning?",
        "What is Artificial Intelligence?",
        "Tell me a joke.",
        "Hello Assistant."
    ],
    "answer": [
        "Amazon EC2 is a cloud service that lets you run virtual servers on demand.",
        "Amazon S3 is a storage service where you can store and retrieve any amount of data securely.",
        "AWS Lambda lets you run code without managing servers. You only pay for the compute time you use.",
        "IAM stands for Identity and Access Management. It helps you securely control access to AWS services and resources.",
        "Cloud computing means delivering computing services like servers, storage, databases, and software over the internet.",
        "Python is used for data science, web development, automation, and building AI applications.",
        "Machine learning is a way to teach computers to learn patterns from data and make predictions without being explicitly programmed.",
        "Artificial Intelligence is the simulation of human intelligence in machines that can think, learn, and make decisions.",
        "Sure! Why don’t programmers like nature? Because it has too many bugs.",
        "Hi Abhishek, how can I help you today?"
    ]
}


df = pd.DataFrame(data)
df

# -----------------------------
# 3. Tokenize Questions
# -----------------------------
def tokenize(text):
    return nltk.word_tokenize(text.lower())

df["tokens"] = df["question"].apply(tokenize)

df[["question", "tokens"]]


# -----------------------------
# 4. Train Word2Vec Model
# -----------------------------
model = Word2Vec(
    sentences=df["tokens"],
    vector_size=50,
    window=3,
    min_count=1,
    workers=4
)

print("Word2Vec model trained successfully")

# -----------------------------
# 5. Convert Sentence into Vector
# -----------------------------
def sentence_vector(sentence):
    words = tokenize(sentence)
    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(50)

    return np.mean(vectors, axis=0)

question_vectors = np.array([
    sentence_vector(q) for q in df["question"]
])

print(question_vectors.shape)

# -----------------------------
# 6. Find Best Answer
# -----------------------------
def get_answer(user_question):
    user_vector = sentence_vector(user_question).reshape(1, -1)

    similarities = cosine_similarity(user_vector, question_vectors)[0]

    best_index = np.argmax(similarities)
    best_score = similarities[best_index]

    if best_score < 0.30:
        return "Sorry, I could not understand your question."

    return df.iloc[best_index]["answer"]
# -----------------------------
# 7. Convert Answer Text to Audio
# -----------------------------
def speak_colab(text):
    filename = "answer.mp3"

    if os.path.exists(filename):
        os.remove(filename)

    tts = gTTS(text=text, lang="en")
    tts.save(filename)

    display(Audio(filename, autoplay=True))
# -----------------------------
# 8. Test Chatbot in Colab
# -----------------------------
user_input = input("Ask AWS question: ")

answer = get_answer(user_input)

print("Bot:", answer)

speak_colab(answer)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Word2Vec model trained successfully
(10, 50)
Ask AWS question:  Hello Assistant
Bot: Hi Abhishek, how can I help you today?
